# Processing
The preprocessing relies on the PSML dataset, which provides a dataloader
## The dataloader

In [ ]:
import pandas as pd

raw_path = "/media/lll/T9/PSML/Minute-level Load and Renewable/CAISO_zone_1_.csv"
raw = pd.read_csv(raw_path)

print("raw data points (3 years in Caizo zone 1):", len(raw), raw.shape, len(raw)/43201)

In [ ]:
import sys
sys.path.append("/home/lll/thesis_lf_cnn/baselines/benchmark/Code")

from dataloader import TimeSeriesLoader

# just dont reload the kernel 
loader = TimeSeriesLoader(task = 'forecasting', root = "/media/lll/T9/PSML")

In [ ]:
train_loader, test_loader = loader.load(batch_size=512, shuffle=False) # no shuffle for time series (maybe this is not correct)

In [ ]:
print("length of the train data: ", len(train_loader.dataset))
print("length of the test data: ", len(test_loader.dataset))

In [ ]:
print(len(train_loader))

In [ ]:
for batch in train_loader:
    print("len(batch):", len(batch))
    for i, item in enumerate(batch):
        print(f"batch[{i}] shape:", item.shape)
    print("if following tensor contains only 0 the mask is not needed: ")
    print(sum(batch[2]==0))


In [ ]:
print("Number of training batches: ", len(train_loader))
print("Number of testing batches: ", len(test_loader))


In [ ]:
feature_dim = batch[0].shape[1]

for T in range(1, 200):
    if feature_dim % T == 0:
        print("Fensterlänge:", T, "Features pro Schritt:", feature_dim // T)


batch[0].shape  
print(batch[0][0][:50])    
print(batch[0][0][-50:])    


In [ ]:
## get column names 
import pandas as pd
cols = pd.read_csv("/media/lll/T9/PSML/processed_dataset/forecasting/CAISO_zone_1_2018.csv", nrows = 0)
print(cols.columns)

## Trying to understand where I lose most of my training data

### What does the loading of the processed dataset do?

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("/home/lll/thesis_lf_cnn/baselines/benchmark/Code")

data = pd.read_csv("/media/lll/T9/PSML/processed_dataset/forecasting/CAISO_zone_1_2018.csv")

train_flag = data['train_flag'].to_numpy()
all_train_idx = np.sort(np.argwhere(train_flag == 1).reshape([-1]))
print("training data before sliding window:", len(all_train_idx))

sliding_window = 120
training_index = all_train_idx[sliding_window:]
print("train data after sliding window:", len(training_index))

from BenchmarkModel.LoadForecasting.processing import task_prediction_horizon, external_feature_names

history_column_names = []
target_val_column_names = []
for task_name, task_prediction_horizon_list in task_prediction_horizon.items():
    history_column_names.append(f'y{task_name[0]}_t')
    for horizon_val in task_prediction_horizon_list:
        target_val_column_names.append(f'y{task_name[0]}_t+{horizon_val}(val)')

target_flag_column_names = [c.replace("val", "flag") for c in target_val_column_names]

training_validation_target_flag = data[target_flag_column_names].to_numpy()[training_index[sliding_window:]]
print("training data before flags matrix:", training_validation_target_flag.shape[0])

 # We use all flag filters at the same time which results in losing a big chunk of data
 # Maybe it would be smart to solve the different Horizons one by one and set specialized flag for each one to train each on max data
selected_index = np.argwhere(np.prod(training_validation_target_flag, axis=-1) == 1).reshape([-1])
print("training data after flags matrix (selected_index):", len(selected_index))

only_one_index = np.argwhere(training_validation_target_flag[:, 0] == 1).reshape([-1])
print("training data after first flags filter only (only_one_index):", len(only_one_index))



In [ ]:
history_column_names = []
target_val_column_names = []

for task_name, horizon_list in task_prediction_horizon.items():
    history_column_names.append(f"y{task_name[0]}_t")
    for h in horizon_list:
        target_val_column_names.append(f"y{task_name[0]}_t+{h}(val)")

target_flag_column_names = [c.replace("(val)", "(flag)") for c in target_val_column_names]


training_validation_target_flag = data[target_flag_column_names].to_numpy()[training_index[sliding_window:]]

num_samples = training_validation_target_flag.shape[0]

print("all training data:", num_samples)
print("number of targets:", training_validation_target_flag.shape[1])
print("After every flag filter")

result = []

for i, col in enumerate(target_flag_column_names):
    valid_mask = (training_validation_target_flag[:, i] == 1)
    valid_count = valid_mask.sum()
    result.append((col, valid_count))
    print(f"{col:35s} : {valid_count:6d} / {num_samples}")

# combined filter

all_valid = np.prod(training_validation_target_flag, axis=-1) == 1
print("datapoints left for all filters combined")
print(f"datapoints: {all_valid.sum():6d} / {num_samples}")